In [6]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("kacpergregorowicz/house-plant-species") + '\\house_plant_species'

In [7]:
import torch
import torch.nn as nn
from model import PlantCNN
from torchvision.models import resnet18

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# model = resnet18(weights=None)
# model.fc = nn.Linear(model.fc.in_features, 100)
model = PlantCNN(num_classes=47)
model.load_state_dict(torch.load("best_model.pth", map_location=device))
model = model.to(device)
model.eval()


C:\Users\Semyon\AppData\Local\Temp\ipykernel_3864\2232430949.py:11: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("best_model.pth", map_loca

PlantCNN(
  (conv1): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv3): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn3): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (adaptive_pool): AdaptiveAvgPool2d(output_size=(4, 4))
  (fc1): Linear(in_features=2048, out_features=512, bias=True)
  (dropout): Dropout(p=0.5, inplace=False)
  (fc2): Linear(in_features=512, out_features=47, bias=True)
)

In [8]:
from AircraftDataset import AircraftDataset
from torchvision import transforms
from torch.utils.data import DataLoader
from data_loader import load_data
import torch

transform_test = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

_, _, test_dataset, classes = load_data(path, 1, transform_test) 
test_loader = DataLoader(test_dataset, batch_size=16, pin_memory=True, num_workers=4)


In [9]:
from tqdm import tqdm


correct, total = 0, 0
with torch.no_grad():
    for images, labels in tqdm(test_loader):
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f"Test accuracy: {correct / total:.4f}")


In [10]:
from OneImage import predict_image


class_mapping = classes

# result1 = predict_image("data/fgvc-aircraft/images/0063281.jpg", model, class_mapping, device)
# result2 = predict_image("data/3.jpg", model, class_mapping, device)

# print("Predicted class:", result2)


In [11]:
import tkinter as tk
from tkinter import filedialog
from PIL import Image, ImageTk
import torch
from torchvision import transforms
import os
import random

# Модель, трансформ, класс-маппинг
model.eval()
class_to_idx = class_mapping 
idx_to_class = {v: k for k, v in class_to_idx.items()}

transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# UI
root = tk.Tk()
root.title("Plant Classifier")

image_label = tk.Label(root)
image_label.pack()

result_label = tk.Label(root, text="", font=("Arial", 14))
result_label.pack()

similar_frame = tk.Frame(root)
similar_frame.pack()

def load_image():
    file_path = filedialog.askopenfilename()
    if not file_path:
        return

    # Оригинал
    img = Image.open(file_path).convert('RGB')
    img_resized = img.resize((480, 320))
    tk_img = ImageTk.PhotoImage(img_resized)
    image_label.configure(image=tk_img)
    image_label.image = tk_img

    # Предсказание
    input_tensor = transform(img).unsqueeze(0).to(device)
    with torch.no_grad():
        output = model(input_tensor)
        predicted_idx = output.argmax(1).item()
        predicted_class = idx_to_class[predicted_idx]

    result_label.config(text=f"Распознанное растение: {predicted_class}")

    # Похожие изображения
    for widget in similar_frame.winfo_children():
        widget.destroy()

    class_df = test_dataset.data[test_dataset.data['Classes'] == predicted_class]
    sample_paths = class_df.sample(min(5, len(class_df)))['filename'].tolist()

    for path in sample_paths:
        full_path = os.path.join(test_dataset.img_dir, path)
        sim_img = Image.open(full_path).convert('RGB').resize((128, 128))
        tk_sim = ImageTk.PhotoImage(sim_img)
        lbl = tk.Label(similar_frame, image=tk_sim)
        lbl.image = tk_sim
        lbl.pack(side="left", padx=5)

tk.Button(root, text="Загрузить изображение", command=load_image).pack(pady=10)

root.mainloop()


Exception in Tkinter callback
Traceback (most recent call last):
  File "C:\ProgramData\miniconda3\Lib\tkinter\__init__.py", line 1968, in __call__
    return self.func(*args)
           ^^^^^^^^^^^^^^^^
  File "C:\Users\Semyon\AppData\Local\Temp\ipykernel_3864\2108139328.py", line 59, in load_image
    class_df = test_dataset.data[test_dataset.data['Classes'] == predicted_class]
               ^^^^^^^^^^^^^^^^^
AttributeError: 'Subset' object has no attribute 'data'
Exception in Tkinter callback
Traceback (most recent call last):
  File "C:\ProgramData\miniconda3\Lib\tkinter\__init__.py", line 1968, in __call__
    return self.func(*args)
           ^^^^^^^^^^^^^^^^
  File "C:\Users\Semyon\AppData\Local\Temp\ipykernel_3864\2108139328.py", line 59, in load_image
    class_df = test_dataset.data[test_dataset.data['Classes'] == predicted_class]
               ^^^^^^^^^^^^^^^^^
AttributeError: 'Subset' object has no attribute 'data'
Exception in Tkinter callback
Traceback (most recent cal